In [ ]:
import sys                      # Permet de configurer les chemins des imports
from pathlib import Path        # Permet de manipuler les chemins de fichiers

project_root = Path.cwd()       # Récupère le dossier de travail de Jupyter

if not (project_root / "scripts").is_dir():  # Si on n'est pas déjà à la racine
    project_root = project_root.parent.parent  # Remonte depuis notebooks/fine_tuning

assert (project_root / "scripts").is_dir(), "Vérifie le dossier de travail."

if str(project_root) not in sys.path:       # Évite d'ajouter deux fois le chemin
    sys.path.insert(0, str(project_root))   # Rend les scripts du projet importables

print("Racine du projet :", project_root)  # Affiche le chemin trouvé

In [ ]:
import torch                           # Bibliothèque utilisée pour entraîner le modèle
from transformers import set_seed      # Fonction qui fixe les graines aléatoires

SEED = 42                              # Même valeur pour reproduire nos expériences
set_seed(SEED)                         # Fixe le hasard de Python, NumPy et PyTorch

cuda_available = torch.cuda.is_available()  # Vérifie si PyTorch peut utiliser CUDA

print("Version PyTorch :", torch.__version__)  # Affiche la version installée
print("CUDA disponible :", cuda_available)     # Affiche True ou False

if cuda_available:                            # Si un GPU CUDA est accessible
    print("GPU :", torch.cuda.get_device_name(0))  # Affiche le nom du premier GPU

In [ ]:
# Chemin des annotations dans l'environnement d'origine
DATA_DIR = Path("/mnt/imported/data/sav-label-studio/Tasks-intention-transaction/OutputTasks")

assert DATA_DIR.is_dir(), f"Dossier introuvable : {DATA_DIR}"  # Vérifie le chemin

data_files = sorted(
    path for path in DATA_DIR.rglob("*") if path.is_file()
)  # Liste les fichiers du dossier et de ses sous-dossiers

print("Nombre de fichiers :", len(data_files))  # Compte les fichiers trouvés

for path in data_files[:5]:   # Prend les cinq premiers fichiers
    print(path.name)          # Affiche leur nom

In [ ]:
import json  # Permet de lire les fichiers JSON

assert data_files, "Aucun fichier trouvé."  # Vérifie que la liste n'est pas vide

example_path = data_files[0]  # Choisit le premier fichier de la liste

with example_path.open("r", encoding="utf-8") as file:  # Ouvre le fichier en lecture
    raw_example = json.load(file)                    # Charge son contenu en Python

print(
    json.dumps(raw_example, indent=2, ensure_ascii=False)
)  # Affiche le contenu lisiblement, en conservant les accents

In [ ]:
def get_choices(example, field_name):          # Recherche un champ d'annotation
    for annotation in example["result"]:      # Parcourt les annotations du message
        if annotation["from_name"] == field_name:  # Repère le champ demandé
            return annotation["value"].get("choices", [])  # Renvoie ses choix

    return []  # Renvoie une liste vide si le champ n'existe pas


text = raw_example["task"]["data"]["message"]  # Récupère le texte du message
intents = get_choices(raw_example, "intents")  # Récupère toutes ses intentions
intent_types = get_choices(raw_example, "type_intent")  # Récupère son type

print("Texte :", text)                        # Affiche le message
print("Intentions :", intents)                # Affiche la liste des intentions
print("Types d'intention :", intent_types)    # Affiche les choix du champ type

In [ ]:
def prepare_example(example):  # Transforme une annotation en un exemple simplifié
    if example["was_cancelled"]:  # Vérifie si l'annotation a été annulée
        return None               # Indique qu'on ne garde pas cet exemple

    review = get_choices(example, "review_decision")  # Récupère la décision de relecture

    if review != ["Accepter"]:  # Écarte les annotations non acceptées
        return None

    data = example["task"]["data"]  # Récupère les données du message
    intents = get_choices(example, "intents")  # Conserve toutes les intentions
    intent_types = get_choices(example, "type_intent")  # Récupère les types sélectionnés

    if not intents:  # Vérifie qu'au moins une intention est renseignée
        raise ValueError("Cet exemple accepté n'a aucune intention.")

    if len(intent_types) != 1:  # Vérifie qu'il y a exactement un type d'intention
        raise ValueError(f"Un seul type attendu, trouvé : {intent_types}")

    assimilated = get_choices(example, "Assimilated_by_Mode")  # Garde cette information pour le filtrage

    return {  # Construit le dictionnaire représentant notre exemple
        "text": data["message"],              # Texte donné au modèle
        "intents": intents,                   # Liste des intentions à prédire
        "type_intent": intent_types[0],       # Unique type à prédire
        "split": data["split"],               # Groupe d'origine : train ou test
        "Assimilated_by_Mode": assimilated[0] if assimilated else None,  # None si absent
    }


prepared_example = prepare_example(raw_example)  # Applique la fonction au fichier déjà lu
print(prepared_example)                         # Affiche le résultat

In [ ]:
def prepare_example(example):  # Transforme une annotation en un exemple simplifié
    if example["was_cancelled"]:  # Vérifie si l'annotation a été annulée
        return None               # Indique qu'on ne garde pas cet exemple

    review = get_choices(example, "review_decision")  # Récupère la décision de relecture

    if review != ["Accepter"]:  # Écarte les annotations non acceptées
        return None

    data = example["task"]["data"]  # Récupère les données du message
    intents = get_choices(example, "intents")  # Conserve toutes les intentions
    intent_types = get_choices(example, "type_intent")  # Récupère les types sélectionnés

    if not intents:  # Vérifie qu'au moins une intention est renseignée
        raise ValueError("Cet exemple accepté n'a aucune intention.")

    if len(intent_types) != 1:  # Vérifie qu'il y a exactement un type d'intention
        raise ValueError(f"Un seul type attendu, trouvé : {intent_types}")

    assimilated = get_choices(example, "Assimilated_by_Mode")  # Garde cette information pour le filtrage

    return {  # Construit le dictionnaire représentant notre exemple
        "text": data["message"],              # Texte donné au modèle
        "intents": intents,                   # Liste des intentions à prédire
        "type_intent": intent_types[0],       # Unique type à prédire
        "split": data["split"],               # Groupe d'origine : train ou test
        "Assimilated_by_Mode": assimilated[0] if assimilated else None,  # None si absent
    }


prepared_example = prepare_example(raw_example)  # Applique la fonction au fichier déjà lu
print(prepared_example)                         # Affiche le résultat

In [ ]:
examples = []        # Contiendra les exemples conservés
excluded_count = 0   # Compte les annotations annulées ou non acceptées

for path in data_files:  # Parcourt les chemins des fichiers trouvés précédemment
    with path.open("r", encoding="utf-8") as file:  # Ouvre le fichier courant
        raw_example = json.load(file)            # Charge son annotation JSON

    example = prepare_example(raw_example)  # Extrait les champs et vérifie l'annotation

    if example is None:         # Si la fonction a écarté cette annotation
        excluded_count += 1    # Ajoute 1 au compteur
        continue               # Passe directement au fichier suivant

    examples.append(example)   # Ajoute l'exemple conservé à notre liste

print("Exemples conservés :", len(examples))  # Affiche la taille de notre liste
print("Annotations écartées :", excluded_count)  # Affiche le nombre d'exclusions

assert examples, "Aucun exemple conservé : vérifie les fichiers et les filtres."

print(examples[0])  # Affiche le premier exemple conservé

In [ ]:
from collections import Counter  # Permet de compter les occurrences de chaque valeur

split_counts = Counter(example["split"] for example in examples)  # Compte les exemples par groupe

type_counts = Counter(
    example["type_intent"] for example in examples
)  # Compte les exemples pour chaque type d'intention

assimilation_counts = Counter(
    example["Assimilated_by_Mode"] for example in examples
)  # Compte les différentes valeurs de ce champ, y compris None

multilabel_count = sum(
    len(example["intents"]) > 1 for example in examples
)  # Compte les exemples qui possèdent plusieurs intentions

print("Répartition par groupe :", dict(split_counts))  # Affiche les effectifs train/test
print("Types d'intention :", dict(type_counts))        # Affiche les effectifs par type
print("Assimilation :", dict(assimilation_counts))     # Affiche les informations de filtrage
print("Exemples avec plusieurs intentions :", multilabel_count)
print("Nombre total d'exemples :", len(examples))

In [ ]:
EXCLUDE_ASSIMILATED = True  # Active l'exclusion des exemples marqués comme assimilés

if EXCLUDE_ASSIMILATED:  # Applique le filtre uniquement s'il est activé
    filtered_examples = [
        example for example in examples  # Parcourt les exemples chargés
        if example["Assimilated_by_Mode"] != "Assimilés par le modèle"  # Garde les autres
    ]
else:
    filtered_examples = examples.copy()  # Copie la liste sans exclure d'exemples

removed_count = len(examples) - len(filtered_examples)  # Calcule le nombre d'exclusions

print("Exemples retirés :", removed_count)          # Affiche l'effet du filtre
print("Exemples restants :", len(filtered_examples))  # Affiche la nouvelle taille

assert filtered_examples, "Le filtrage a supprimé tous les exemples."

In [ ]:
split_names = {example["split"] for example in filtered_examples}  # Récupère les valeurs distinctes

unexpected_splits = split_names - {"train", "test"}  # Repère les valeurs non prévues

if unexpected_splits:  # Arrête l'exécution si un groupe n'est pas reconnu
    raise ValueError(f"Valeurs de split inattendues : {unexpected_splits}")

train_examples = [
    example for example in filtered_examples
    if example["split"] == "train"  # Garde les exemples destinés à l'entraînement
]

test_examples = [
    example for example in filtered_examples
    if example["split"] == "test"  # Garde les exemples destinés au test
]

assert train_examples, "Le groupe train est vide."  # Vérifie la présence de données d'entraînement
assert test_examples, "Le groupe test est vide."    # Vérifie la présence de données de test

print("Exemples train :", len(train_examples))  # Affiche la taille du groupe train
print("Exemples test :", len(test_examples))    # Affiche la taille du groupe test

In [ ]:
from sklearn.model_selection import train_test_split  # Permet de séparer une liste en deux groupes

VALIDATION_RATIO = 0.20  # Réserve 20 % du train d'origine pour la validation

train_pool = [
    example for example in filtered_examples
    if example["split"] == "train"  # Repart toujours du train d'origine
]

train_examples, validation_examples = train_test_split(
    train_pool,                  # Données à répartir entre entraînement et validation
    test_size=VALIDATION_RATIO,   # Proportion réservée à la validation
    random_state=SEED,            # Rend la séparation reproductible
    shuffle=True,                # Mélange les exemples avant de les séparer
)

print("Entraînement :", len(train_examples))       # Exemples utilisés pour apprendre
print("Validation :", len(validation_examples))   # Exemples utilisés pour choisir le modèle
print("Test :", len(test_examples))               # Exemples réservés à l'évaluation finale

train_examples = [
    example for example in filtered_examples
    if example["split"] == "train"  # Utilise tout le train d'origine pour entraîner
]

validation_examples = test_examples  # Utilise le test comme validation, comme dans l'original

print("Entraînement :", len(train_examples))
print("Validation :", len(validation_examples))
print("Test :", len(test_examples))

In [ ]:
from collections import Counter  # Compte les occurrences de chaque intention

def count_intents(examples):  # Calcule les effectifs pour un groupe d'exemples
    counts = Counter()       # Commence avec un compteur vide

    for example in examples:               # Parcourt les messages
        for intent in set(example["intents"]):  # Prend chaque intention du message une seule fois
            counts[intent] += 1            # Ajoute un exemple pour cette intention

    return counts  # Renvoie les effectifs obtenus


train_counts = count_intents(train_examples)            # Compte dans le train
validation_counts = count_intents(validation_examples)  # Compte dans la validation

all_intents = sorted(
    set(train_counts) | set(validation_counts)
)  # Réunit les intentions des deux groupes et les trie alphabétiquement

for intent in all_intents:  # Affiche les effectifs de chaque intention
    print(
        f"{intent} : "
        f"train={train_counts[intent]}, "
        f"validation={validation_counts[intent]}"
    )

In [ ]:
train_texts = {
    example["text"] for example in train_examples
}  # Récupère les textes distincts du train

validation_texts = {
    example["text"] for example in validation_examples
}  # Récupère les textes distincts de la validation

test_texts = {
    example["text"] for example in test_examples
}  # Récupère les textes distincts du test

train_validation_overlap = train_texts & validation_texts  # Textes communs au train et à la validation
train_test_overlap = train_texts & test_texts              # Textes communs au train et au test
validation_test_overlap = validation_texts & test_texts    # Textes communs à la validation et au test

print("Textes communs train / validation :", len(train_validation_overlap))
print("Textes communs train / test :", len(train_test_overlap))
print("Textes communs validation / test :", len(validation_test_overlap))

In [ ]:
intent_names = sorted(
    intent for intent in train_counts if intent != "Autre"
)  # Liste les intentions présentes dans le train, sauf "Autre"

intent2id = {
    intent: index for index, intent in enumerate(intent_names)
}  # Associe chaque intention à un numéro

id2intent = {
    index: intent for intent, index in intent2id.items()
}  # Construit la correspondance inverse

num_intents = len(intent_names)  # Nombre de sorties nécessaires pour prédire les intentions

assert num_intents > 0, "Aucune intention à apprendre en dehors de 'Autre'."

print("Nombre d'intentions :", num_intents)

for intent, index in intent2id.items():  # Affiche les correspondances
    print(f"{index} → {intent}")

In [ ]:
type_names = sorted({
    example["type_intent"] for example in train_examples
})  # Récupère les types présents dans le train, sans doublons, puis les trie

type2id = {
    intent_type: index for index, intent_type in enumerate(type_names)
}  # Associe chaque type à un numéro

id2type = {
    index: intent_type for intent_type, index in type2id.items()
}  # Permet de retrouver le nom d'un type à partir de son numéro

num_types = len(type_names)  # Nombre de sorties nécessaires pour prédire le type

print("Nombre de types :", num_types)

for intent_type, index in type2id.items():  # Affiche les correspondances
    print(f"{index} → {intent_type}")

In [ ]:
known_intents = set(intent2id) | {"Autre"}  # Intentions connues, avec "Autre" autorisé
known_types = set(type2id)                 # Types d'intention connus

groups_to_check = {
    "validation": validation_examples,  # Groupe utilisé pendant l'entraînement
    "test": test_examples,              # Groupe utilisé pour l'évaluation finale
}

for group_name, group_examples in groups_to_check.items():  # Parcourt les deux groupes
    for example in group_examples:  # Vérifie chaque exemple
        unknown_intents = set(example["intents"]) - known_intents  # Cherche les intentions inconnues

        if unknown_intents:  # Signale les intentions absentes de notre correspondance
            raise ValueError(
                f"{group_name} : intentions inconnues : {sorted(unknown_intents)}"
            )

        if example["type_intent"] not in known_types:  # Vérifie également le type
            raise ValueError(
                f"{group_name} : type inconnu : {example['type_intent']}"
            )

print("Vérification terminée : toutes les étiquettes sont connues.")

In [ ]:
def encode_labels(example):  # Convertit les étiquettes d'un exemple en nombres
    intent_labels = [0.0] * num_intents  # Initialise toutes les intentions à « absente »

    for intent in example["intents"]:  # Parcourt les intentions du message
        if intent != "Autre":         # "Autre" laisse le vecteur rempli de zéros
            index = intent2id[intent]  # Retrouve la position de cette intention
            intent_labels[index] = 1.0  # Marque cette intention comme présente

    type_label = type2id[example["type_intent"]]  # Convertit le type en entier

    return {
        "labels": intent_labels,          # Cible pour la tête qui prédit les intentions
        "labels_type_intents": type_label,  # Cible pour la tête qui prédit le type
    }


example = train_examples[0]             # Prend le premier exemple d'entraînement
encoded_labels = encode_labels(example)  # Convertit ses étiquettes

print("Intentions :", example["intents"])       # Affiche les intentions d'origine
print("Type :", example["type_intent"])         # Affiche le type d'origine
print("Cibles numériques :", encoded_labels)    # Affiche leur représentation numérique

In [ ]:
from datasets import Dataset, DatasetDict  # Importe les structures de données Hugging Face

dataset = DatasetDict({
    "train": Dataset.from_list(train_examples),            # Convertit la liste d'entraînement
    "validation": Dataset.from_list(validation_examples),  # Convertit la liste de validation
    "test": Dataset.from_list(test_examples),              # Convertit la liste de test
})

dataset = dataset.remove_columns(
    ["split", "Assimilated_by_Mode"]
)  # Retire les informations de préparation qui ne serviront pas au modèle

print(dataset)              # Affiche les colonnes et le nombre d'exemples par groupe
print(dataset["train"][0])  # Affiche le premier exemple du train

In [ ]:
dataset = dataset.map(
    encode_labels,  # Fonction qui convertit les étiquettes d'un exemple
    batched=False,  # Lui transmet un seul exemple à la fois
)

print("Intentions :", dataset["train"][0]["intents"])  # Étiquettes originales
print("Vecteur cible :", dataset["train"][0]["labels"])  # Intentions encodées

print("Type :", dataset["train"][0]["type_intent"])  # Type original
print("Type encodé :", dataset["train"][0]["labels_type_intents"])  # Numéro du type

In [ ]:
from transformers import AutoTokenizer  # Charge le tokenizer adapté au modèle

MODEL_PATH = "/mnt/modelhub/ModelHub-model-huggingface-almanach/camembertav2-base/main"  # Dossier du modèle
MAX_LENGTH = 512  # Nombre maximal de tokens par message, tokens spéciaux compris

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH  # Lit les fichiers du tokenizer dans ce dossier
)

text = dataset["train"][0]["text"]  # Récupère le premier message d'entraînement

tokenized_example = tokenizer(
    text,                   # Texte à convertir
    truncation=True,        # Coupe le message s'il dépasse la longueur maximale
    max_length=MAX_LENGTH,  # Fixe cette longueur maximale
)

print("Texte :", text)  # Affiche le message original
print("Premiers identifiants :", tokenized_example["input_ids"][:20])  # Affiche les 20 premiers IDs
print(
    "Premiers tokens :",
    tokenizer.convert_ids_to_tokens(tokenized_example["input_ids"][:20]),
)  # Affiche les morceaux de texte correspondant à ces IDs
print("Nombre de tokens :", len(tokenized_example["input_ids"]))  # Compte les tokens obtenus

In [ ]:
def tokenize_batch(batch):  # Reçoit un groupe d'exemples
    return tokenizer(
        batch["text"],          # Liste des textes du groupe
        truncation=True,        # Coupe les messages trop longs
        max_length=MAX_LENGTH,  # Limite chaque message à 512 tokens
        padding=False,          # Le remplissage sera effectué lors de la création des batches
    )


encoded_dataset = dataset.map(
    tokenize_batch,  # Applique notre fonction de tokenisation
    batched=True,   # Lui transmet plusieurs exemples à la fois
    remove_columns=["text", "intents", "type_intent"],  # Retire les colonnes devenues inutiles au modèle
)

print(encoded_dataset)              # Affiche la structure des datasets encodés
print(encoded_dataset["train"][0])  # Affiche le premier exemple prêt pour la suite

In [ ]:
from transformers import DataCollatorWithPadding  # Prépare les batches avec un padding adapté

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,  # Utilise notamment l'identifiant du token de padding
    padding=True,        # Complète jusqu'à la longueur du plus long message du batch
    return_tensors="pt",  # Renvoie des tenseurs PyTorch
)

batch_size = min(4, len(encoded_dataset["train"]))  # Prend jusqu'à 4 exemples pour la démonstration

batch_examples = [
    encoded_dataset["train"][i] for i in range(batch_size)
]  # Récupère les exemples individuellement

example_batch = data_collator(batch_examples)  # Assemble les exemples en un batch

for name, tensor in example_batch.items():  # Parcourt les entrées et les cibles du batch
    print(name, ":", tensor.shape, tensor.dtype)  # Affiche leurs dimensions et leur type numérique

In [ ]:
from sklearn.metrics import (
    accuracy_score,   # Mesure la proportion de prédictions entièrement correctes
    f1_score,         # Combine précision et rappel
    precision_score,  # Mesure la justesse des étiquettes prédites positives
    recall_score,     # Mesure la proportion d'étiquettes positives retrouvées
)


def compute_classification_metrics(true_labels, predicted_labels, prefix):
    return {
        f"{prefix}_f1_micro": f1_score(
            true_labels, predicted_labels, average="micro", zero_division=0
        ),  # Calcule le F1 en regroupant les résultats de toutes les classes

        f"{prefix}_f1_macro": f1_score(
            true_labels, predicted_labels, average="macro", zero_division=0
        ),  # Calcule le F1 par classe, puis leur moyenne

        f"{prefix}_precision_macro": precision_score(
            true_labels, predicted_labels, average="macro", zero_division=0
        ),  # Moyenne des précisions calculées par classe

        f"{prefix}_recall_macro": recall_score(
            true_labels, predicted_labels, average="macro", zero_division=0
        ),  # Moyenne des rappels calculés par classe

        f"{prefix}_accuracy": accuracy_score(
            true_labels, predicted_labels
        ),  # Proportion d'exemples dont toutes les étiquettes sont correctes
    }

In [ ]:
import numpy as np  # Permet notamment de chercher la position du score maximal

INTENT_THRESHOLD = 0.5  # Seuil à partir duquel une intention est considérée comme présente


def compute_metrics(eval_prediction):  # Reçoit les sorties du modèle et les cibles attendues
    intent_logits, type_logits = eval_prediction.predictions  # Récupère les scores des deux têtes
    true_intents, true_types = eval_prediction.label_ids      # Récupère les deux cibles

    intent_probabilities = torch.sigmoid(
        torch.as_tensor(intent_logits)
    ).numpy()  # Transforme les logits des intentions en valeurs entre 0 et 1

    predicted_intents = (
        intent_probabilities >= INTENT_THRESHOLD
    ).astype(int)  # Transforme chaque probabilité en 0 ou 1

    predicted_types = np.argmax(
        type_logits, axis=-1
    )  # Sélectionne le type ayant le score le plus élevé pour chaque message

    metrics = compute_classification_metrics(
        true_intents, predicted_intents, prefix="intent"
    )  # Calcule les scores des intentions

    type_metrics = compute_classification_metrics(
        true_types, predicted_types, prefix="type_intent"
    )  # Calcule les scores du type d'intention

    metrics.update(type_metrics)  # Réunit les scores des deux tâches dans un dictionnaire

    return metrics  # Transmet les scores au Trainer

In [ ]:
from scripts.models.modeling_deberta_v2 import DebertaV2MultiTasksConfig  # Configuration personnalisée du projet

config = DebertaV2MultiTasksConfig.from_pretrained(
    MODEL_PATH  # Charge la configuration du modèle préentraîné
)

config.problem_type = "multi_label_classification"  # Autorise plusieurs intentions par message

config.num_labels = num_intents  # Fixe le nombre de sorties de la tête des intentions
config.id2label = id2intent      # Associe chaque position à son nom d'intention
config.label2id = intent2id      # Associe chaque intention à sa position

config.num_type_intent = num_types  # Fixe le nombre de sorties de la tête des types
config.id2type_intent = id2type      # Associe chaque numéro à son type
config.type_intent2id = type2id      # Associe chaque type à son numéro

print("Problème :", config.problem_type)
print("Nombre d'intentions :", config.num_labels)
print("Nombre de types :", config.num_type_intent)

In [ ]:
from scripts.models.modeling_deberta_v2 import DebertaV2ForMultitasks  # Modèle personnalisé du projet

model = DebertaV2ForMultitasks.from_pretrained(
    MODEL_PATH,     # Dossier contenant les poids préentraînés
    config=config,  # Configuration adaptée à nos intentions et types
)

print("Tête des intentions :", model.intents_classifier)  # Affiche la dernière couche des intentions
print("Tête des types :", model.intent_type_classifier)   # Affiche la dernière couche des types

In [ ]:
from datetime import datetime  # Permet de dater chaque entraînement

run_name = datetime.now().strftime("%Y%m%d_%H%M%S_%f")  # Date et heure, avec les microsecondes

run_dir = project_root / "outputs" / "fine_tuning_multi_head" / run_name  # Dossier de ce nouvel essai

run_dir.mkdir(
    parents=True,    # Crée les dossiers parents si nécessaire
    exist_ok=False,  # Arrête avec une erreur si ce dossier existe déjà
)

final_model_dir = run_dir / "best_model"  # Prévoit un emplacement pour la sauvegarde finale

print("Dossier de cet entraînement :", run_dir)
print("Future sauvegarde finale :", final_model_dir)

In [ ]:
from transformers import TrainingArguments  # Regroupe les paramètres utilisés par le Trainer

training_args = TrainingArguments(
    output_dir=str(run_dir / "checkpoints"),  # Checkpoints de ce nouvel entraînement uniquement
    logging_dir=str(run_dir / "logs"),       # Dossier prévu pour ses logs

    num_train_epochs=30,             # Prévoit 30 passages sur les données d'entraînement
    per_device_train_batch_size=4,   # Traite 4 exemples par batch d'entraînement et par appareil
    per_device_eval_batch_size=4,    # Traite 4 exemples par batch d'évaluation et par appareil
    gradient_accumulation_steps=1,   # Met à jour les poids après chaque batch d'entraînement

    learning_rate=2e-5,  # Fixe le taux d'apprentissage initial du programme d'entraînement
    warmup_ratio=0.1,   # Consacre les premiers 10 % des étapes à augmenter progressivement ce taux
    weight_decay=0.01,  # Applique une régularisation qui pénalise les poids élevés

    eval_strategy="epoch",     # Évalue sur la validation après chaque epoch
    save_strategy="epoch",     # Sauvegarde un checkpoint après chaque epoch
    logging_strategy="epoch",  # Affiche les statistiques d'entraînement après chaque epoch

    load_best_model_at_end=True,             # Recharge le meilleur checkpoint à la fin
    metric_for_best_model="intent_f1_macro",  # Utilise ce score de validation pour le sélectionner
    greater_is_better=True,                  # Considère qu'un score plus élevé est meilleur
    save_total_limit=2,                      # Limite le nombre de checkpoints conservés

    label_names=["labels", "labels_type_intents"],  # Indique les deux cibles et leur ordre

    seed=SEED,       # Fixe la graine utilisée pour les opérations aléatoires
    data_seed=SEED,  # Fixe la graine utilisée pour l'échantillonnage des données
    fp16=False,     # Désactive l'entraînement en précision float16
    bf16=False,     # Désactive l'entraînement en précision bfloat16
    report_to="none",  # Désactive l'envoi des métriques aux outils de suivi externes
)

In [ ]:
from transformers import Trainer  # Gère l'entraînement et l'évaluation

trainer = Trainer(
    model=model,                         # Modèle préentraîné avec nos deux têtes
    args=training_args,                   # Paramètres et chemins de ce nouvel entraînement
    train_dataset=encoded_dataset["train"],        # Données utilisées pour apprendre
    eval_dataset=encoded_dataset["validation"],    # Données utilisées pour évaluer à chaque epoch
    processing_class=tokenizer,           # Fournit le tokenizer et active le collator avec padding par défaut
    compute_metrics=compute_metrics,      # Fonction qui calcule les scores des deux tâches
)

print("Dossier des checkpoints :", trainer.args.output_dir)  # Vérifie la destination des sauvegardes
print("Exemples d'entraînement :", len(trainer.train_dataset))
print("Exemples de validation :", len(trainer.eval_dataset))

In [ ]:
train_result = trainer.train()  # Lance l'entraînement et les évaluations prévues

print(train_result.metrics)  # Affiche le bilan global de l'entraînement

In [ ]:
test_metrics = trainer.evaluate(
    eval_dataset=encoded_dataset["test"],  # Utilise les exemples du groupe test
    metric_key_prefix="test",              # Préfixe les noms des résultats par "test_"
)

for name, value in test_metrics.items():  # Parcourt les résultats obtenus
    print(f"{name} : {value:.4f}")        # Affiche chaque valeur avec quatre décimales